# ICT-15b -- Sensitivity Canonicity (Huang 2019 transpose au zoo ICT)

*See [#7288](https://github.com/jsboige/CoursIA/issues/7288) -- Part of Epic #4588 (strate 5 : theorie fondatrice cross-substrat).*

## La question

Le theoreme de Huang (2019) etablit en 2 pages que pour toute fonction booleenne `f : {0,1}^n -> {0,1}` :

    s(f) >= sqrt(deg(f))

ou `s(f)` est la sensibilite (max nb de voisins ou `f` bascule) et `deg(f)` le degre polynomial comme representant sur l'hypercube. La preuve est **spectrale** : la matrice de signes `A` verifie `A^2 = n * Id` (entrelacement de Cauchy).

ICT-15 a falsifie le scalaire universel : `Phi / F / K` ne se reduisent pas l'un a l'autre. ICT-15b transpose : **existe-t-il un scalaire LOCAL dont une fonction simple borne les scalaires GLOBAUX du zoo ICT ?**

## La conjecture ICT-15b (enoncee AVANT les experiences)

Pour une fonction d'etat `f` sur le graphe de transition Markovien d'une trajectoire ICT :

    s_max(f) >= sqrt(deg_proxy(f))

ou `s_max(f) = max_x s_x(f)` (sensibilite maximale locale) et `deg_proxy(f)` est le degre **moyen du voisinage** du graphe de transition (proxy conservatif : degre local). C'est la transposition la plus directe du theoreme de Huang au zoo ICT.

**Statut epistemique** : la trajectoire ICT n'est PAS une fonction booleenne statique. L'hypercube, le degre polynomial, la structure produit -- tout cela se perd. La conjecture est **testable mais pas theorique** : un verdict negatif (`inconsistent`) sur plusieurs substrats serait un **resultat** (information irreductiblement globale).

## Substrats testes

| Substrat | Strate | Source trajectoire | Fonction d'etat `f` |
|---|---|---|---|
| **S1 -- Tri morphogenetique** | Strate 1 | `SelfSortingArray` (Zhang/Goldstein/Levin 2025) | indicatrice tableau trie |
| **S2 -- Regime bistable** | Strate 2 | `bistable.GrazingModel` | regime actuel (haut/bas) |
| **S3 -- Opinion majoritaire** | Strate 3 | marche aleatoire sur graphe complet `K_n` | majorite courante |
| **S4 -- Motif causal** | Strate 5 | `epsilon_machine` surrogate | etat causal courant |

Toutes les experiences utilisent `ict.spectral.transition_graph` + `ict.sensitivity.huang_conjecture_test` (modules merges dans le PR #7330).


In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')

from ict import spectral, sensitivity, bistable
from ict.spectral import transition_graph, current_matrix, signed_adjacency, laplacian_spectrum, spectral_gap
from ict.sensitivity import local_sensitivity, sensitivity_distribution, huang_conjecture_test

print('ict.spectral OK, transition_graph:', transition_graph)
print('ict.sensitivity OK, huang_conjecture_test:', huang_conjecture_test)
print('numpy:', np.__version__)


ict.spectral OK, transition_graph: <function transition_graph at 0x000002A71ED5ADE0>
ict.sensitivity OK, huang_conjecture_test: <function huang_conjecture_test at 0x000002A71ED5B560>
numpy: 2.4.4


## Utilitaire : trajectoire -> sequence de labels

Les modules `transition_graph` / `huang_conjecture_test` attendent une **sequence d'etats encodes en labels** (entiers 0..n_symbols-1) plutot que les etats natifs (tableaux, regimes, etc.). On definit un helper qui reduit un etat natif a son label canonique.


In [2]:
def trajectory_labels(states, key_fn):
    label_map = {}
    labels = []
    for s in states:
        k = key_fn(s)
        if k not in label_map:
            label_map[k] = len(label_map)
        labels.append(label_map[k])
    return labels, len(label_map), label_map


def summarize_substrat(name, states, key_fn, f_native, label_to_native):
    labels, n_symbols, _ = trajectory_labels(states, key_fn)
    f_labels = lambda lab: f_native(label_to_native[lab])
    print('--- ' + name + ' ---')
    print('  n_symbols (vocabulaire) :', n_symbols)
    print('  len(trajectoire)         :', len(labels))
    dist = sensitivity_distribution(labels, n_symbols, f_labels)
    print('  sensibilite : max=' + format(dist['max'], '.2f') + ', mean=' + format(dist['mean'], '.2f') + ', std=' + format(dist['std'], '.2f') + ', p95=' + format(dist['p95'], '.2f'))
    test = huang_conjecture_test(labels, n_symbols, f_labels)
    print('  s_max       : ' + format(test['s_max'], '.3f'))
    print('  deg_proxy   : ' + format(test['deg_proxy'], '.3f'))
    print('  threshold   : ' + format(test['threshold'], '.3f') + ' (= sqrt(deg_proxy))')
    print('  ratio       : ' + format(test['ratio'], '.3f') + '  (s_max / threshold)')
    print('  verdict     : ' + test['verdict'])
    print()
    return dist, test


## S1 -- Tri morphogenetique (strate 1, Zhang/Goldstein/Levin 2025)

Substrat canonique de la serie ICT : un tableau de cellules qui se trient **elles-memes** par regles locales (algotypes `bubble` / `insertion`). La trajectoire = suite des configurations du tableau. La fonction d'etat `f` = indicatrice le tableau est-il trie.

**Hypothese implicite** : un tableau qui se trie est un systeme qui converge vers un attracteur. La sensibilite sur cet attracteur est nulle (`f` constante). La conjecture devrait etre `inconclusive` (trajectoire trop courte apres convergence) ou `consistent` sur la phase transitoire.


In [3]:
from ict import self_sorting

rng = np.random.default_rng(0)
values = list(rng.permutation(8))
ssa = self_sorting.SelfSortingArray(values, seed=42)

states_s1 = []
N_STEPS_S1 = 600
for _ in range(N_STEPS_S1):
    states_s1.append(tuple(ssa.values))
    if not ssa.has_move():
        break
    ssa.step()

print('  S1 : ' + str(len(states_s1)) + ' pas, converge=' + str(not ssa.has_move()) + ', valeurs finales=' + str(ssa.values))

_labels_s1, n_s1, _label_map_s1 = trajectory_labels(states_s1, lambda s: s)
label_to_state_s1 = {v: k for k, v in _label_map_s1.items()}
f_native_s1 = lambda natif: int(all(natif[i] <= natif[i + 1] for i in range(len(natif) - 1)))

dist_s1, test_s1 = summarize_substrat('S1 (tri)', states_s1, lambda s: s, f_native_s1, label_to_state_s1)


  S1 : 61 pas, converge=True, valeurs finales=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
--- S1 (tri) ---
  n_symbols (vocabulaire) : 13
  len(trajectoire)         : 61
  sensibilite : max=12.00, mean=1.85, std=2.93, p95=5.40
  s_max       : 12.000
  deg_proxy   : 0.565
  threshold   : 0.752 (= sqrt(deg_proxy))
  ratio       : 15.968  (s_max / threshold)
  verdict     : consistent



## S2 -- Regime bistable (strate 2, modele de paturage)

`ict.bistable.GrazingModel` : un systeme a 2 regimes stables (haut/bas) avec bifurcation pli. La trajectoire = suite des regimes visites. La fonction d'etat `f` = indicatrice regime haut.

**Hypothese** : un systeme bistable reste longtemps dans un regime, bascule rarement. La sensibilite devrait etre elevee sur les points de bascule, faible ailleurs. Conjecture : `consistent` ou `inconclusive` selon la longueur de la trajectoire apres les bascules.


In [4]:
grazing = bistable.GrazingModel()
states_s2 = grazing.simulate_sde(c=1.6, x0=8.0, sigma=0.3, dt=0.05, T=2000, seed=7)

x_high_threshold = 5.0
n_haut = int(np.sum(states_s2 > x_high_threshold))
n_bas = int(np.sum(states_s2 <= x_high_threshold))
print('  S2 : ' + str(len(states_s2)) + ' pas, repartition regimes (seuil x=' + str(x_high_threshold) + ') : haut=' + str(n_haut) + ', bas=' + str(n_bas))

_labels_s2, n_s2, _label_map_s2 = trajectory_labels(states_s2, lambda x: round(float(x), 3))
label_to_state_s2 = {v: k for k, v in _label_map_s2.items()}
f_native_s2 = lambda natif: int(natif > x_high_threshold)

dist_s2, test_s2 = summarize_substrat('S2 (regime)', states_s2, lambda x: round(float(x), 3), f_native_s2, label_to_state_s2)


  S2 : 2000 pas, repartition regimes (seuil x=5.0) : haut=2000, bas=0
--- S2 (regime) ---
  n_symbols (vocabulaire) : 980
  len(trajectoire)         : 2000
  sensibilite : max=0.00, mean=0.00, std=0.00, p95=0.00
  s_max       : 0.000
  deg_proxy   : 0.996
  threshold   : 0.998 (= sqrt(deg_proxy))
  ratio       : 0.000  (s_max / threshold)
  verdict     : inconsistent



## S3 -- Opinion majoritaire (Sznajd simplifie)

Modele Sznajd simplifie : sur un graphe complet `K_n`, chaque pas tire une paire uniforme ; si les deux agents sont d'accord, tous leurs voisins adoptent leur opinion. La trajectoire = suite des majorites (somme des opinions).

**Hypothese** : le systeme converge vers un consensus (tous `+1` ou tous `-1`). La sensibilite de la majorite devrait etre elevee en phase transitoire, nulle apres consensus.


In [5]:
def simulate_sznajd(n_agents=50, n_steps=1500, seed=0):
    rng = np.random.default_rng(seed)
    opinions = rng.choice([-1, 1], size=n_agents)
    states = []
    for _ in range(n_steps):
        states.append(int(np.sum(opinions > 0)))
        i, j = rng.choice(n_agents, size=2, replace=False)
        if opinions[i] == opinions[j]:
            new_op = opinions[i]
            for k in range(n_agents):
                if k != i and k != j:
                    opinions[k] = new_op
    return states

states_s3 = simulate_sznajd()
print('  S3 : ' + str(len(states_s3)) + ' pas, majorites uniques (echantillon) : ' + str(sorted(set(states_s3))[:5]))

_labels_s3, n_s3, _label_map_s3 = trajectory_labels(states_s3, lambda x: x)
label_to_state_s3 = {v: k for k, v in _label_map_s3.items()}
f_native_s3 = lambda natif: int(natif >= 25)

dist_s3, test_s3 = summarize_substrat('S3 (opinion)', states_s3, lambda x: x, f_native_s3, label_to_state_s3)


  S3 : 1500 pas, majorites uniques (echantillon) : [0, 27]
--- S3 (opinion) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 1500
  sensibilite : max=1.00, mean=1.00, std=0.00, p95=1.00
  s_max       : 1.000
  deg_proxy   : 0.250
  threshold   : 0.500 (= sqrt(deg_proxy))
  ratio       : 2.000  (s_max / threshold)
  verdict     : consistent



## S4 -- Motif causal (goldmine / even-odd cycle)

Substrat plus riche : on genere une sequence avec une structure causale cachee (2 etats caches avec probabilites d'emission differentes), puis on regarde la sensibilite de l'etat observe.

**Hypothese** : la sensibilite devrait discriminer les transitions entre regimes d'emission.


In [6]:
def goldmine_sequence(n_steps=2000, seed=0):
    rng = np.random.default_rng(seed)
    states = []
    causal = 0
    for _ in range(n_steps):
        if causal == 0:
            emit = 'A' if rng.random() < 0.9 else 'B'
        else:
            emit = 'B' if rng.random() < 0.9 else 'A'
        states.append(emit)
        causal = 1 - causal if rng.random() < 0.3 else causal
    return states

seq_s4 = goldmine_sequence()
print('  S4 : ' + str(len(seq_s4)) + ' pas, emissions uniques : ' + str(sorted(set(seq_s4))))

_labels_s4, n_s4, _label_map_s4 = trajectory_labels(seq_s4, lambda x: x)
label_to_state_s4 = {v: k for k, v in _label_map_s4.items()}
f_native_s4 = lambda natif: int(natif == 'A')

dist_s4, test_s4 = summarize_substrat('S4 (motif)', seq_s4, lambda x: x, f_native_s4, label_to_state_s4)


  S4 : 2000 pas, emissions uniques : ['A', 'B']
--- S4 (motif) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 2000
  sensibilite : max=1.00, mean=1.00, std=0.00, p95=1.00
  s_max       : 1.000
  deg_proxy   : 0.344
  threshold   : 0.587 (= sqrt(deg_proxy))
  ratio       : 1.705  (s_max / threshold)
  verdict     : consistent



## Synthese des verdicts (par substrat)

Tableau recapitulatif avec pandas pour la lisibilite.


In [7]:
import pandas as pd

rows = []
for name, test in [('S1 (tri)', test_s1), ('S2 (regime)', test_s2), ('S3 (opinion)', test_s3), ('S4 (motif)', test_s4)]:
    rows.append({
        'Substrat': name,
        's_max': round(test['s_max'], 3),
        'deg_proxy': round(test['deg_proxy'], 3),
        'threshold': round(test['threshold'], 3),
        'ratio': round(test['ratio'], 3),
        'verdict': test['verdict'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

verdicts = [r['verdict'] for r in rows]
n_consistent = verdicts.count('consistent')
n_inconsistent = verdicts.count('inconsistent')
n_inconclusive = verdicts.count('inconclusive')
print()
print('Verdict global : ' + str(n_consistent) + ' consistent / ' + str(n_inconsistent) + ' inconsistent / ' + str(n_inconclusive) + ' inconclusive sur ' + str(len(rows)) + ' substrats')
print()
print('Interpretation :')
if n_inconsistent > 0:
    print('  La conjecture ICT-15b (s_max >= sqrt(deg_proxy)) est REJETEE sur ' + str(n_inconsistent) + ' substrat(s).')
    print('  C est un resultat : la sensibilite locale ne suffit pas a borner le zoo ICT,')
    print('  l integration exige une information irreductiblement globale.')
elif n_consistent == len(rows):
    print('  La conjecture ICT-15b TIENT sur tous les substrats. Mais consistent n est pas')
    print('  une preuve : c est une observation empirique sur 4 substrats particuliers.')
else:
    print('  Resultat mixte : la conjecture tient sur certains substrats, echoue sur d autres.')


    Substrat  s_max  deg_proxy  threshold  ratio      verdict
    S1 (tri)     12      0.565      0.752 15.968   consistent
 S2 (regime)      0      0.996      0.998  0.000 inconsistent
S3 (opinion)      1      0.250      0.500  2.000   consistent
  S4 (motif)      1      0.344      0.587  1.705   consistent

Verdict global : 3 consistent / 1 inconsistent / 0 inconclusive sur 4 substrats

Interpretation :
  La conjecture ICT-15b (s_max >= sqrt(deg_proxy)) est REJETEE sur 1 substrat(s).
  C est un resultat : la sensibilite locale ne suffit pas a borner le zoo ICT,
  l integration exige une information irreductiblement globale.


## Verdict honnete et pont Lean

**Quel que soit le verdict empirique, ICT-15b NE RESOUD PAS la question du scalaire universel** : la conjecture transposee n'a pas la portee du theoreme de Huang (qui est une borne **pour toute fonction booleenne**). Ici on teste une seule fonction `f` par substrat ; la generalisation demanderait un echantillonnage.

**Pont formel** : le lake `sensitivity_lean` (Lean 4, WSL) heberge le **vrai** theoreme de Huang sur l'hypercube booleen (`A^2 = n * Id`, entrelacement de Cauchy, `s >= sqrt(deg)`) avec **0 sorry** dans les modules principaux (`/workspace/sensitivity_lean/Spectral/SignMatrix.lean`, `SensitivityDegree.lean`). C'est la **jambe formelle** ; ICT-15b est la **jambe empirique** sur des donnees non-booleennes. Les deux cohabitent : ICT-15b demontre ou refute la **transposition**, pas le theoreme lui-meme.

References :
- Theorem original : H. Huang, *Induced subgraphs of hypercubes and a proof of the Sensitivity Conjecture*, 2019.
- Lean 4 companion : `MyIA.AI.Notebooks/SymbolicAI/Lean/sensitivity_lean/` (PR #7330).
- ICT-15b scope : voir issue [#7288](https://github.com/jsboige/CoursIA/issues/7288).


## Exercices

Trois exercices d'application (regle C.1 : stubs sans `raise NotImplementedError`, notebook executable end-to-end).

### Exercice 1 -- Fonction d'etat personnalisee sur S3

Reprends la trajectoire S3 (Sznajd) et teste la conjecture ICT-15b avec une **autre** fonction d'etat : `f(label) = 1 si majorite >= 80% des agents`. Commente la difference de verdict avec la majorite simple (50%).


In [8]:
def f_s3_strict(label, majority_ref, label_to_state):
    return 1 if majority_ref[label_to_state[label]] >= 40 else 0

f_native_s3_strict = lambda natif: int(natif >= 40)
dist_s3_strict, test_s3_strict = summarize_substrat('S3 strict (>=80%)', states_s3, lambda x: x, f_native_s3_strict, label_to_state_s3)

print('Verdict majorite simple (>=50%) : ' + test_s3['verdict'])
print('Verdict majorite stricte (>=80%) : ' + test_s3_strict['verdict'])
# TODO etudiant : commenter la difference


--- S3 strict (>=80%) ---
  n_symbols (vocabulaire) : 2
  len(trajectoire)         : 1500
  sensibilite : max=0.00, mean=0.00, std=0.00, p95=0.00
  s_max       : 0.000
  deg_proxy   : 0.250
  threshold   : 0.500 (= sqrt(deg_proxy))
  ratio       : 0.000  (s_max / threshold)
  verdict     : inconsistent

Verdict majorite simple (>=50%) : consistent
Verdict majorite stricte (>=80%) : inconsistent


### Exercice 2 -- Strategie de seuil sur la sensibilite

Implemente une strategie qui choisit le **label** avec la sensibilite locale maximale et verifie empiriquement que ce label est effectivement sur un point de bascule dans la trajectoire (utilise `local_sensitivity` + inspection manuelle de la trajectoire).


In [9]:
labels_s1b, n_sym_s1b, label_map_s1b = trajectory_labels(states_s1, lambda s: s)
label_to_state_s1b = {v: k for k, v in label_map_s1b.items()}
f_native_s1b = lambda natif: int(all(natif[i] <= natif[i + 1] for i in range(len(natif) - 1)))
f_labels_s1b = lambda lab: f_native_s1b(label_to_state_s1b[lab])
s_x_s1b = local_sensitivity(labels_s1b, n_sym_s1b, f_labels_s1b)

visited_s1b = sorted(set(labels_s1b))
best_s1b = max(visited_s1b, key=lambda lab: int(s_x_s1b[min(lab, len(s_x_s1b) - 1)]))
s_value_s1b = int(s_x_s1b[min(best_s1b, len(s_x_s1b) - 1)])
print('Label le plus sensible parmi ceux visites : ' + str(best_s1b) + ' (s_x = ' + str(s_value_s1b) + ')')
print('Etat correspondant    : ' + str(label_to_state_s1b[best_s1b]))
print('Trajectoire autour du pic (5 voisins du pic) :')
for i in range(max(0, best_s1b - 2), min(len(states_s1), best_s1b + 3)):
    marker = ' <-- PIC' if labels_s1b[i] == best_s1b else ''
    print('  t=' + str(i) + ' (label=' + str(labels_s1b[i]) + ') : ' + str(label_to_state_s1b[labels_s1b[i]]) + marker)
# TODO etudiant : le pic est-il sur un point de bascule (avant / apres tri complet) ?


Label le plus sensible parmi ceux visites : 12 (s_x = 12)
Etat correspondant    : (np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7))
Trajectoire autour du pic (5 voisins du pic) :
  t=10 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=11 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=12 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=13 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))
  t=14 (label=6) : (np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(5), np.int64(1), np.int64(7))


### Exercice 3 -- Lean comme oracle

Le theoreme de Huang est formalise dans `sensitivity_lean/Spectral/SensitivityDegree.lean` avec **0 sorry**. Sans ouvrir le Lean, reproduis la **borne** `s >= sqrt(deg)` sur une fonction booleenne simple : la fonction **OR** sur 4 variables (`deg = 1`, `s = 1`) et la fonction **parite** sur 4 variables (`deg = 4`, `s = 4`). Verifie que les deux respectent la borne.


In [10]:
def boolean_sensitivity(truth_table):
    n = len(truth_table).bit_length() - 1
    assert len(truth_table) == 2 ** n
    s_max = 0
    for x_int in range(2 ** n):
        f_x = truth_table[x_int]
        n_flips = 0
        for b in range(n):
            y_int = x_int ^ (1 << b)
            if truth_table[y_int] != f_x:
                n_flips += 1
        s_max = max(s_max, n_flips)
    return s_max


or_tt = [int(bin(x).count('1') >= 1) for x in range(2 ** 4)]
parite_tt = [int(bin(x).count('1') % 2 == 1) for x in range(2 ** 4)]

s_or = boolean_sensitivity(or_tt)
s_parite = boolean_sensitivity(parite_tt)

print('OR(n=4)     : s = ' + str(s_or) + ', sqrt(deg=1) = ' + format(1 ** 0.5, '.2f') + ', borne respectee : ' + str(s_or >= 1 ** 0.5))
print('Parite(n=4) : s = ' + str(s_parite) + ', sqrt(deg=4) = ' + format(4 ** 0.5, '.2f') + ', borne respectee : ' + str(s_parite >= 4 ** 0.5))

# Verdict honnete : borne TIENT sur les 2 cas triviaux (et le Lean la tient sur tous les cas).
# Huang 2019 montre qu'elle tient pour TOUTE fonction booleenne ; ICT-15b transpose au zoo ICT,
# ou on n'a PAS l'equivalent de la structure de l'hypercube.


OR(n=4)     : s = 4, sqrt(deg=1) = 1.00, borne respectee : True
Parite(n=4) : s = 4, sqrt(deg=4) = 2.00, borne respectee : True
